# Leverage, Margin Trading, and Returns


Margin trading means using your own capital plus borrowed funds to control a larger position.

Leverage is the exposure multiplier (for example, 2x leverage means controlling $2 of assets for every $1 of equity).

Because the position is larger relative to your equity, leverage amplifies both gains and losses.

This notebook compares simple returns and log returns under leverage, and shows why leverage should be applied to simple returns, not by directly scaling log returns.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = '{:,.6f}'.format


## 1) One-Period Upside Example (100 -> 110, 50% margin, 2x leverage)


In [3]:
# Inputs
initial_price = 100.0
final_price = 110.0
margin_requirement = 0.50
leverage = 1 / margin_requirement  # 2x

# Equity needed to control one share under leverage
margin_equity = initial_price / leverage

# P&L for controlling one share
profit = final_price - initial_price

# Returns
unlevered_simple_return = final_price / initial_price - 1
levered_simple_return = profit / margin_equity

unlevered_log_return = np.log(final_price / initial_price)
# Levered log return is based on equity growth under leverage
levered_log_return = np.log((margin_equity + profit) / margin_equity)

print('Upside example inputs:')
print(f'  Initial price:       {initial_price:.2f}')
print(f'  Final price:         {final_price:.2f}')
print(f'  Margin requirement:  {margin_requirement:.0%}')
print(f'  Leverage:            {leverage:.1f}x')
print(f'  Margin/equity used:  {margin_equity:.2f}')
print()
print('Calculated returns:')
print(f'  Absolute profit:             {profit:.2f}')
print(f'  Unlevered simple return:     {unlevered_simple_return:.2%}')
print(f'  Levered simple return:       {levered_simple_return:.2%}')
print(f'  Unlevered log return:        {unlevered_log_return:.6f}')
print(f'  Levered log return:          {levered_log_return:.6f}')


Upside example inputs:
  Initial price:       100.00
  Final price:         110.00
  Margin requirement:  50%
  Leverage:            2.0x
  Margin/equity used:  50.00

Calculated returns:
  Absolute profit:             10.00
  Unlevered simple return:     10.00%
  Levered simple return:       20.00%
  Unlevered log return:        0.095310
  Levered log return:          0.182322


## 2) Relationship Tests


In [4]:
simple_relation_left = levered_simple_return
simple_relation_right = unlevered_simple_return * leverage

log_relation_left = levered_log_return
log_relation_right = unlevered_log_return * leverage

simple_holds = np.isclose(simple_relation_left, simple_relation_right)
log_holds = np.isclose(log_relation_left, log_relation_right)

print('Simple-return scaling test:')
print(f'  Levered simple return:              {simple_relation_left:.6f}')
print(f'  Unlevered simple return * leverage: {simple_relation_right:.6f}')
print(f'  Relationship holds? {simple_holds}')
print()
print('Log-return scaling test:')
print(f'  Levered log return:                 {log_relation_left:.6f}')
print(f'  Unlevered log return * leverage:    {log_relation_right:.6f}')
print(f'  Relationship holds? {log_holds}')


Simple-return scaling test:
  Levered simple return:              0.200000
  Unlevered simple return * leverage: 0.200000
  Relationship holds? True

Log-return scaling test:
  Levered log return:                 0.182322
  Unlevered log return * leverage:    0.190620
  Relationship holds? False


In [5]:
upside_summary = pd.DataFrame({
    'initial_price': [initial_price],
    'final_price': [final_price],
    'leverage': [leverage],
    'margin': [margin_equity],
    'profit': [profit],
    'unlevered_simple_return': [unlevered_simple_return],
    'levered_simple_return': [levered_simple_return],
    'unlevered_log_return': [unlevered_log_return],
    'levered_log_return': [levered_log_return],
    'unlevered_log_return_times_leverage': [unlevered_log_return * leverage],
})

upside_summary


,initial_price,final_price,leverage,margin,profit,unlevered_simple_return,levered_simple_return,unlevered_log_return,levered_log_return,unlevered_log_return_times_leverage
0,100.000000,110.000000,2.000000,50.000000,10.000000,0.100000,0.200000,0.095310,0.182322,0.190620


### Interpretation

- Simple returns scale linearly with leverage in this setup (`levered simple = leverage × unlevered simple`).
- Log returns do not scale linearly this way.
- Therefore, leverage should be applied to simple returns, not by directly multiplying log returns.


## 3) Downside Example (100 -> 90, same 50% margin and 2x leverage)


In [6]:
initial_price_down = 100.0
final_price_down = 90.0
margin_equity_down = initial_price_down / leverage

profit_down = final_price_down - initial_price_down

unlevered_simple_down = final_price_down / initial_price_down - 1
levered_simple_down = profit_down / margin_equity_down

unlevered_log_down = np.log(final_price_down / initial_price_down)
levered_log_down = np.log((margin_equity_down + profit_down) / margin_equity_down)

print('Downside example results:')
print(f'  Profit/Loss:                 {profit_down:.2f}')
print(f'  Unlevered simple return:     {unlevered_simple_down:.2%}')
print(f'  Levered simple return:       {levered_simple_down:.2%}')
print(f'  Unlevered log return:        {unlevered_log_down:.6f}')
print(f'  Levered log return:          {levered_log_down:.6f}')


Downside example results:
  Profit/Loss:                 -10.00
  Unlevered simple return:     -10.00%
  Levered simple return:       -20.00%
  Unlevered log return:        -0.105361
  Levered log return:          -0.223144


Leverage magnifies losses too: a -10% unlevered simple return becomes -20% with 2x leverage in this one-period setup.


In [7]:
comparison_table = pd.DataFrame({
    'scenario': ['Upside (100 -> 110)', 'Downside (100 -> 90)'],
    'initial_price': [initial_price, initial_price_down],
    'final_price': [final_price, final_price_down],
    'leverage': [leverage, leverage],
    'margin': [margin_equity, margin_equity_down],
    'profit': [profit, profit_down],
    'unlevered_simple_return': [unlevered_simple_return, unlevered_simple_down],
    'levered_simple_return': [levered_simple_return, levered_simple_down],
    'unlevered_log_return': [unlevered_log_return, unlevered_log_down],
    'levered_log_return': [levered_log_return, levered_log_down],
})

comparison_table


,scenario,initial_price,final_price,leverage,margin,profit,unlevered_simple_return,levered_simple_return,unlevered_log_return,levered_log_return
0,Upside (100 -> 110),100.000000,110.000000,2.000000,50.000000,10.000000,0.100000,0.200000,0.095310,0.182322
1,Downside (100 -> 90),100.000000,90.000000,2.000000,50.000000,-10.000000,-0.100000,-0.200000,-0.105361,-0.223144


## 4) Visualization (Upside Case)


In [ ]:
labels = [
    'Unlevered\nSimple',
    'Levered\nSimple',
    'Unlevered\nLog',
    'Levered\nLog',
]
values = [
    unlevered_simple_return,
    levered_simple_return,
    unlevered_log_return,
    levered_log_return,
]

plt.figure(figsize=(8, 5))
plt.bar(labels, values)
plt.axhline(0, linewidth=1)
plt.ylabel('Return')
plt.title('Upside Case: Return Comparison Under 2x Leverage')
plt.tight_layout()
plt.show()


## Key Takeaways

- Margin trading means using borrowed funds.
- Leverage increases exposure relative to your own equity.
- Leverage amplifies gains and losses.
- Simple returns can be scaled by leverage (in this one-period setup).
- Log returns should not simply be multiplied by leverage.
- Misapplying log-return scaling in leveraged settings can lead to analysis errors.
